# 🌍 Portfolio Project — Drought Risk Mapping for the MENA Region

**Author:** Ines Gasmi, PhD  
**Focus:** Climate risk, geospatial analytics, machine learning, drought early warning  
**Core stack:** Python · pandas · NumPy · matplotlib · scikit-learn  
**Target upgrade path:** MODIS MOD13A3 · CHIRPS rainfall · Sentinel-2 · XGBoost · Folium/Plotly

---

## Why this project matters

Drought is one of the most damaging climate hazards across the MENA region. It reduces crop productivity, increases pressure on water resources, and weakens rural resilience. This notebook demonstrates how satellite-derived vegetation signals can be transformed into a simple and explainable drought-risk workflow.

## What this notebook demonstrates

This portfolio notebook shows the full logic of an end-to-end geospatial ML project:

- build a vegetation stress signal from time-series NDVI
- derive anomaly-based drought classes
- engineer climate and seasonality features
- train an interpretable machine learning baseline
- communicate results through charts and spatial views

## Current project status

This version is a **prototype** built with simulated satellite-like data so the workflow runs end-to-end without external downloads. The structure is intentionally designed so it can be upgraded to **real MODIS and rainfall data** in the next iteration.


## Notebook Roadmap

1. **Project setup**
2. **Data design and prototype assumptions**
3. **NDVI climatology and anomaly detection**
4. **Drought classification**
5. **Feature engineering**
6. **Machine learning baseline**
7. **Results and interpretation**
8. **Portfolio-ready next steps**


---
## 1. Project Setup
This section imports the packages used across the notebook and defines plotting defaults.


In [ ]:
# Install required packages (run once)
# !pip install numpy pandas geopandas rasterio matplotlib seaborn scikit-learn xarray folium

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ML
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Set plot style
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Reproducibility
np.random.seed(42)

print('✅ All imports successful')

---
## 2. Data Design and Prototype Assumptions

### What this section does
To keep the notebook runnable and easy to review, the current version generates **synthetic but realistic drought-related data** for the MENA region.

### Why this is acceptable in a prototype
This allows the notebook to demonstrate:
- time-series feature engineering
- anomaly-based drought logic
- multi-class classification
- portfolio-grade visualization

### What will change in the production version
The synthetic inputs should be replaced with:
- **MODIS MOD13A3** for monthly NDVI
- **CHIRPS** for rainfall
- optional **Sentinel-2** for finer-resolution pilot zones

### Real-data integration note
The pipeline structure is reusable, but replacing simulated inputs with observed data will still require proper ingestion, quality filtering, temporal aggregation, and spatial alignment.


In [ ]:
# ── Simulation parameters ──────────────────────────────────────────────────
N_MONTHS   = 120        # 10 years: 2013–2022
N_PIXELS   = 2000       # spatial units (pixels/grid cells)
N_REGIONS  = 5          # sub-regions with different aridity

REGION_NAMES = [
    'Coastal Mediterranean',   # higher NDVI, wetter
    'Atlas / Tell Atlas',
    'Semi-arid Steppe',
    'Pre-Saharan Transition',
    'Saharan Zone',            # lowest NDVI, driest
]

# Base NDVI by region (realistic values for North Africa)
region_base_ndvi = [0.45, 0.32, 0.22, 0.14, 0.06]

# Assign pixels to regions
pixel_regions = np.random.choice(N_REGIONS, size=N_PIXELS, p=[0.12, 0.18, 0.28, 0.25, 0.17])

# ── Build NDVI time series ─────────────────────────────────────────────────
months = np.arange(N_MONTHS)

# Drought years: 2015, 2017, 2022 (historically significant in MENA)
drought_years = {24: -0.08, 25: -0.10, 26: -0.07,   # 2015
                 48: -0.09, 49: -0.12, 50: -0.08,   # 2017
                 108: -0.11, 109: -0.14, 110: -0.10} # 2022

ndvi_data = np.zeros((N_MONTHS, N_PIXELS))

for pix in range(N_PIXELS):
    reg = pixel_regions[pix]
    base = region_base_ndvi[reg]

    for t in months:
        # Seasonal cycle (peak in spring for Mediterranean, minimal in Saharan)
        seasonal = base * 0.35 * np.sin(2 * np.pi * (t % 12) / 12 - np.pi/6)
        # Interannual trend (slight greening due to CO2 fertilization)
        trend = 0.0002 * t
        # Random noise
        noise = np.random.normal(0, base * 0.08)
        # Drought shock
        drought_shock = drought_years.get(t, 0) * (1 + (4 - reg) * 0.15)

        ndvi_data[t, pix] = np.clip(
            base + seasonal + trend + noise + drought_shock,
            0.0, 0.95
        )

# Rainfall data (mm/month, correlated with NDVI with lag)
rainfall_data = np.zeros((N_MONTHS, N_PIXELS))
for pix in range(N_PIXELS):
    reg = pixel_regions[pix]
    base_rain = [60, 40, 25, 12, 3][reg]
    for t in months:
        seasonal_rain = base_rain * 0.8 * np.sin(2 * np.pi * (t % 12) / 12)
        drought_shock = drought_years.get(t, 0) * base_rain * 3
        noise = np.random.exponential(base_rain * 0.15)
        rainfall_data[t, pix] = np.clip(base_rain + seasonal_rain + drought_shock + noise, 0, None)

# Land Surface Temperature (°C)
lst_data = np.zeros((N_MONTHS, N_PIXELS))
for pix in range(N_PIXELS):
    reg = pixel_regions[pix]
    base_lst = [22, 26, 30, 36, 42][reg]
    for t in months:
        seasonal_lst = 8 * np.sin(2 * np.pi * (t % 12) / 12)
        drought_boost = -drought_years.get(t, 0) * 15  # hotter during drought
        noise = np.random.normal(0, 1.2)
        lst_data[t, pix] = base_lst + seasonal_lst + drought_boost + noise

# Date index
dates = pd.date_range(start='2013-01', periods=N_MONTHS, freq='MS')

print(f'✅ Simulated {N_MONTHS} months × {N_PIXELS} pixels')
print(f'   NDVI range: {ndvi_data.min():.3f} – {ndvi_data.max():.3f}')
print(f'   Rainfall range: {rainfall_data.min():.1f} – {rainfall_data.max():.1f} mm/month')
print(f'   LST range: {lst_data.min():.1f} – {lst_data.max():.1f} °C')

---
## 3. NDVI Climatology and Anomaly Detection

NDVI measures vegetation greenness and is widely used as a proxy for plant condition.

$$NDVI = \frac{NIR - Red}{NIR + Red}$$

### Why anomalies matter
Raw NDVI alone is not enough because vegetation naturally changes by season.  
A more useful drought signal is the **anomaly**, which compares each month to its long-term average for the same calendar month.

### In this notebook
We calculate:
- monthly climatology
- monthly NDVI anomaly
- smoothed multi-month anomaly for a more stable drought signal


In [ ]:
# ── Long-term monthly climatology (baseline) ───────────────────────────────
ndvi_climatology = np.zeros((12, N_PIXELS))  # mean NDVI for each calendar month
for month in range(12):
    month_indices = [t for t in range(N_MONTHS) if t % 12 == month]
    ndvi_climatology[month] = ndvi_data[month_indices].mean(axis=0)

# ── Calculate anomaly for each timestep ───────────────────────────────────
ndvi_anomaly = np.zeros_like(ndvi_data)
for t in range(N_MONTHS):
    ndvi_anomaly[t] = ndvi_data[t] - ndvi_climatology[t % 12]

# ── 3-month rolling mean anomaly (SPI-style smoothing) ────────────────────
ndvi_anomaly_smooth = pd.DataFrame(ndvi_anomaly).rolling(3, min_periods=1).mean().values

# ── Regional mean time series ──────────────────────────────────────────────
region_ndvi = {}
region_anomaly = {}
for reg in range(N_REGIONS):
    mask = pixel_regions == reg
    region_ndvi[reg] = ndvi_data[:, mask].mean(axis=1)
    region_anomaly[reg] = ndvi_anomaly[:, mask].mean(axis=1)

print('✅ NDVI climatology and anomalies computed')
print(f'   Mean anomaly (all): {ndvi_anomaly.mean():.4f}')
print(f'   Std anomaly (all):  {ndvi_anomaly.std():.4f}')

In [ ]:
# ── Plot 1: Regional NDVI time series with drought periods ─────────────────
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#8B4513']
drought_periods = [(24, 26, '2015'), (48, 50, '2017'), (108, 110, '2022')]

fig, axes = plt.subplots(N_REGIONS, 1, figsize=(14, 12), sharex=True)
fig.suptitle('NDVI Anomaly by Sub-Region — North Africa (2013–2022)',
             fontsize=14, fontweight='bold', y=1.01)

for reg, ax in enumerate(axes):
    anom = region_anomaly[reg]
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')

    # Shade drought periods
    for start, end, label in drought_periods:
        ax.axvspan(start, end, alpha=0.12, color='#C73E1D', zorder=0)
        if reg == 0:
            ax.text((start + end) / 2, anom.max() * 0.9, label,
                    ha='center', fontsize=8, color='#C73E1D', fontweight='bold')

    ax.fill_between(range(N_MONTHS), anom, 0,
                    where=anom < 0, color='#C73E1D', alpha=0.4, label='Below baseline')
    ax.fill_between(range(N_MONTHS), anom, 0,
                    where=anom >= 0, color='#2E86AB', alpha=0.4, label='Above baseline')
    ax.plot(anom, color=colors[reg], linewidth=0.8, alpha=0.7)

    ax.set_ylabel(REGION_NAMES[reg], fontsize=9, labelpad=8)
    ax.set_ylim(-0.15, 0.12)

# X-axis labels (years)
year_ticks = list(range(0, N_MONTHS, 12))
year_labels = [str(2013 + i) for i in range(len(year_ticks))]
axes[-1].set_xticks(year_ticks)
axes[-1].set_xticklabels(year_labels)
axes[-1].set_xlabel('Year', fontsize=10)

# Shared legend
handles = [
    mpatches.Patch(color='#C73E1D', alpha=0.5, label='Below baseline (drought stress)'),
    mpatches.Patch(color='#2E86AB', alpha=0.5, label='Above baseline'),
    mpatches.Patch(color='#C73E1D', alpha=0.15, label='Major drought year'),
]
fig.legend(handles=handles, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.04), fontsize=9)
plt.tight_layout()
plt.savefig('ndvi_anomaly_by_region.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Figure saved: ndvi_anomaly_by_region.png')

---
## 4. Drought Classification

This section converts continuous NDVI anomalies into drought severity classes.

### Classification logic
The notebook uses a simple, explainable rule-based approach with four classes:
- Normal / Wet
- Mild Drought
- Moderate Drought
- Severe Drought

This is useful for a first portfolio version because the logic is transparent and easy to interpret before moving to more advanced labels or external drought indices.


In [ ]:
# ── Domain-informed classification thresholds ──────────────────────────────
# Based on: FAO drought monitoring + NDVI anomaly literature for MENA
THRESHOLDS = {
    0: ('Normal / Wet',    '#2E86AB', 0.0),
    1: ('Mild Drought',    '#F18F01', -0.04),
    2: ('Moderate Drought','#E76F51', -0.08),
    3: ('Severe Drought',  '#C73E1D', -0.14),
}

def classify_drought(anomaly_array):
    """Classify drought severity from NDVI anomaly values."""
    classes = np.zeros_like(anomaly_array, dtype=int)
    classes[anomaly_array < -0.04]  = 1
    classes[anomaly_array < -0.08]  = 2
    classes[anomaly_array < -0.14]  = 3
    return classes

drought_classes = classify_drought(ndvi_anomaly)  # shape: (months, pixels)

# ── Drought frequency (% of months in drought per pixel) ──────────────────
in_drought = (drought_classes >= 1).mean(axis=0) * 100  # % months with any drought
severe_drought = (drought_classes == 3).mean(axis=0) * 100

print('✅ Drought classification complete')
print(f'\nDistribution across all pixel-months:')
for cls, (label, color, _) in THRESHOLDS.items():
    pct = (drought_classes == cls).mean() * 100
    bar = '█' * int(pct / 2)
    print(f'  Class {cls} — {label:<22} {bar} {pct:.1f}%')
print(f'\n  Mean drought frequency per pixel: {in_drought.mean():.1f}% of months')
print(f'  Mean severe drought frequency:    {severe_drought.mean():.1f}% of months')

In [ ]:
# ── Plot 2: Spatial drought frequency map (simulated pixel grid) ───────────
# Arrange pixels into a 2D grid for visualization
GRID_W, GRID_H = 50, 40

# Drought frequency grid
freq_grid = in_drought[:GRID_W * GRID_H].reshape(GRID_H, GRID_W)

# Drought severity for a specific drought year (2022 = months 108-110)
drought_2022 = classify_drought(ndvi_anomaly[109])[:GRID_W * GRID_H].reshape(GRID_H, GRID_W)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Drought Analysis — North Africa', fontsize=14, fontweight='bold')

# Left: drought frequency
im1 = axes[0].imshow(freq_grid, cmap='YlOrRd', vmin=0, vmax=60)
plt.colorbar(im1, ax=axes[0], label='% of months in drought (2013–2022)')
axes[0].set_title('Drought Frequency (any severity)', fontsize=12)
axes[0].set_xlabel('← West          East →')
axes[0].set_ylabel('← South          North →')

# Right: drought severity map (2022 peak)
cmap_drought = mcolors.ListedColormap([t[1] for t in THRESHOLDS.values()])
norm_drought = mcolors.BoundaryNorm([0, 1, 2, 3, 4], cmap_drought.N)

im2 = axes[1].imshow(drought_2022, cmap=cmap_drought, norm=norm_drought)
cbar2 = plt.colorbar(im2, ax=axes[1], ticks=[0.5, 1.5, 2.5, 3.5])
cbar2.set_ticklabels([t[0] for t in THRESHOLDS.values()])
axes[1].set_title('Drought Severity — Peak Month 2022', fontsize=12)
axes[1].set_xlabel('← West          East →')

plt.tight_layout()
plt.savefig('drought_maps.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Figure saved: drought_maps.png')

---
## 5. Feature Engineering for Machine Learning

The next step is to transform the drought monitoring workflow into a supervised learning problem.

### Prediction goal
Predict drought severity class from a combination of:
- vegetation condition
- anomaly history
- rainfall
- land surface temperature
- seasonality
- regional context

### Why this matters for a portfolio
This section shows that the project is not only descriptive geospatial analysis. It also includes a clean ML pipeline with engineered predictors and a defined target.


In [ ]:
# ── Build feature table ────────────────────────────────────────────────────
rows = []
for t in range(2, N_MONTHS):  # start at 2 for rolling features
    for pix in range(0, N_PIXELS, 2):  # subsample for speed
        reg = pixel_regions[pix]
        rows.append({
            'ndvi':               ndvi_data[t, pix],
            'ndvi_anomaly':       ndvi_anomaly[t, pix],
            'ndvi_anom_3m':       ndvi_anomaly[t-2:t+1, pix].mean(),
            'ndvi_anom_6m':       ndvi_anomaly[max(0,t-5):t+1, pix].mean(),
            'rainfall':           rainfall_data[t, pix],
            'rainfall_3m':        rainfall_data[t-2:t+1, pix].sum(),
            'lst':                lst_data[t, pix],
            'lst_anomaly':        lst_data[t, pix] - lst_data[max(0,t-12):t, pix].mean() if t >= 12 else 0,
            'region':             reg,
            'month':              t % 12,
            'drought_class':      drought_classes[t, pix],
        })

df = pd.DataFrame(rows)
df = df.dropna()

print(f'✅ Feature table built: {len(df):,} samples')
print(f'\nClass distribution:')
vc = df['drought_class'].value_counts().sort_index()
for cls, count in vc.items():
    label = THRESHOLDS[cls][0]
    pct = count / len(df) * 100
    print(f'  Class {cls} — {label:<22} {count:>6,} ({pct:.1f}%)')

df.describe().round(3)

In [ ]:
# ── Train / test split ─────────────────────────────────────────────────────
FEATURES = ['ndvi', 'ndvi_anomaly', 'ndvi_anom_3m', 'ndvi_anom_6m',
            'rainfall', 'rainfall_3m', 'lst', 'lst_anomaly',
            'region', 'month']
TARGET = 'drought_class'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

# ── Random Forest ──────────────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    min_samples_leaf=5,
    class_weight='balanced',   # handles class imbalance
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

# ── Evaluation ─────────────────────────────────────────────────────────────
y_pred = rf.predict(X_test)
class_names = [THRESHOLDS[i][0] for i in sorted(THRESHOLDS)]

print('\n── Classification Report ─────────────────────────────────')
print(classification_report(y_test, y_pred, target_names=class_names))

# Cross-validation score
cv_scores = cross_val_score(rf, X, y, cv=5, scoring='f1_weighted', n_jobs=-1)
print(f'Cross-validation F1 (weighted): {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

### Model choice

A **Random Forest** is used here as the baseline model because it is:
- strong on tabular data
- robust to non-linear relationships
- relatively easy to interpret
- a better first benchmark than jumping directly to deep learning

A strong baseline is more valuable in a portfolio than unnecessary complexity.


In [ ]:
# ── Plot 3: Confusion matrix + Feature importance ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Random Forest Model Evaluation', fontsize=14, fontweight='bold')

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            ax=axes[0], cbar_kws={'label': '% of true class'})
axes[0].set_xlabel('Predicted', fontsize=11)
axes[0].set_ylabel('Actual', fontsize=11)
axes[0].set_title('Confusion Matrix (% of actual class)', fontsize=11)
axes[0].tick_params(axis='x', rotation=30)

# Feature importance
feat_imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)
colors_fi = ['#C73E1D' if 'anomaly' in f or f == 'ndvi' else
             '#F18F01' if 'rainfall' in f else
             '#2E86AB' for f in feat_imp.index]

feat_imp.plot(kind='barh', ax=axes[1], color=colors_fi)
axes[1].set_xlabel('Feature Importance (Gini)', fontsize=11)
axes[1].set_title('Feature Importance', fontsize=11)

# Legend for feature importance colors
legend_handles = [
    mpatches.Patch(color='#C73E1D', label='NDVI / vegetation signal'),
    mpatches.Patch(color='#F18F01', label='Rainfall signal'),
    mpatches.Patch(color='#2E86AB', label='Contextual (LST, region, month)'),
]
axes[1].legend(handles=legend_handles, fontsize=9, loc='lower right')

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Figure saved: model_evaluation.png')

---
## 6. Results and Interpretation

This section shifts from prediction to interpretation.  
The goal is to understand how drought evolves over time, where exposure is concentrated, and how vegetation stress relates to rainfall across sub-regions.


In [ ]:
# ── Plot 4: Annual drought severity area (stacked) ─────────────────────────
annual_severity = pd.DataFrame(index=range(2013, 2023))
for cls in range(4):
    annual_vals = []
    for yr in range(10):
        t_start, t_end = yr * 12, (yr + 1) * 12
        pct = (drought_classes[t_start:t_end] == cls).mean() * 100
        annual_vals.append(pct)
    annual_severity[THRESHOLDS[cls][0]] = annual_vals

fig, ax = plt.subplots(figsize=(13, 5))
annual_severity.plot(
    kind='bar', stacked=True, ax=ax,
    color=[THRESHOLDS[i][1] for i in range(4)],
    edgecolor='white', linewidth=0.5
)
ax.set_title('Annual Drought Severity Distribution — North Africa (2013–2022)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('% of pixel-months', fontsize=11)
ax.set_xticklabels(range(2013, 2023), rotation=0)
ax.legend(loc='upper left', fontsize=9, title='Drought Class')
ax.set_ylim(0, 100)

# Annotate drought years
for yr_offset, label in [(2, '2015'), (4, '2017'), (9, '2022')]:
    ax.annotate(f'⚠ {label}', xy=(yr_offset, 92), ha='center',
                fontsize=9, color='#C73E1D', fontweight='bold')

plt.tight_layout()
plt.savefig('annual_drought_severity.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Figure saved: annual_drought_severity.png')

In [ ]:
# ── Plot 5: NDVI vs Rainfall scatter by region ─────────────────────────────
fig, axes = plt.subplots(1, N_REGIONS, figsize=(18, 4), sharey=True)
fig.suptitle('NDVI Anomaly vs Rainfall by Sub-Region', fontsize=13, fontweight='bold')

for reg in range(N_REGIONS):
    ax = axes[reg]
    mask = df['region'] == reg
    sub = df[mask].sample(min(500, mask.sum()), random_state=42)

    scatter = ax.scatter(
        sub['rainfall'], sub['ndvi_anomaly'],
        c=sub['drought_class'], cmap='RdYlGn_r',
        vmin=0, vmax=3, alpha=0.5, s=15, edgecolors='none'
    )
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
    ax.set_title(REGION_NAMES[reg], fontsize=9, fontweight='bold')
    ax.set_xlabel('Rainfall (mm/month)', fontsize=9)
    if reg == 0:
        ax.set_ylabel('NDVI Anomaly', fontsize=9)

# Shared colorbar
cbar = fig.colorbar(scatter, ax=axes, shrink=0.8, pad=0.02,
                    ticks=[0.4, 1.1, 1.9, 2.7])
cbar.set_ticklabels(['Normal', 'Mild', 'Moderate', 'Severe'])
cbar.set_label('Drought Class', fontsize=10)

plt.savefig('ndvi_rainfall_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Figure saved: ndvi_rainfall_scatter.png')

---
## 7. Key Findings

The summary below condenses the main signals from the prototype:
- where drought is more persistent
- which years stand out
- how well the baseline model performs
- which features contribute most to classification

For a hiring manager or reviewer, this is the section that answers:  
**What did the project actually find?**


In [ ]:
# ── Summary statistics ─────────────────────────────────────────────────────
print('=' * 60)
print('  DROUGHT RISK ANALYSIS — KEY FINDINGS')
print('  North Africa | 2013–2022 | Satellite-derived NDVI')
print('=' * 60)

print('\n📍 SPATIAL EXPOSURE')
pct_exposed = (in_drought > 20).mean() * 100
pct_severe  = (severe_drought > 5).mean() * 100
print(f'   Pixels with >20% drought months:        {pct_exposed:.1f}%')
print(f'   Pixels with >5% severe drought months:  {pct_severe:.1f}%')

print('\n📅 DROUGHT YEARS')
for yr_offset, yr_label in [(2, '2015'), (4, '2017'), (9, '2022')]:
    t_start = yr_offset * 12
    pct_d = (drought_classes[t_start:t_start+12] >= 1).mean() * 100
    pct_s = (drought_classes[t_start:t_start+12] == 3).mean() * 100
    print(f'   {yr_label}: {pct_d:.1f}% pixels in drought, {pct_s:.1f}% in severe drought')

print('\n🤖 ML MODEL PERFORMANCE')
from sklearn.metrics import f1_score, accuracy_score
acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred, average='weighted')
print(f'   Accuracy (test set):         {acc:.3f}')
print(f'   F1-score weighted (test):    {f1:.3f}')
print(f'   Cross-val F1 (5-fold):       {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

print('\n🔑 TOP PREDICTIVE FEATURES')
top_feats = feat_imp.sort_values(ascending=False).head(5)
for feat, imp in top_feats.items():
    print(f'   {feat:<25} {imp:.3f}')

print('\n💡 INSIGHT')
print('   3- and 6-month NDVI anomaly (cumulative vegetation stress)')
print('   are stronger predictors than single-month values — consistent')
print('   with domain knowledge that drought develops over seasons, not')
print('   weeks. Rainfall is secondary, particularly in Saharan zones')
print('   where baseline rainfall is near zero.')
print('=' * 60)

---
## 8. Portfolio-Ready Next Steps

### Highest-priority upgrades
1. **Replace simulated inputs with real MODIS MOD13A3 and CHIRPS data**  
   This is the most important step to turn the notebook from a prototype into a credible drought-risk product.

2. **Strengthen the preprocessing pipeline**  
   Add data ingestion, QA filtering, temporal aggregation, and spatial harmonization.

3. **Benchmark Random Forest against XGBoost**  
   Compare a stronger tabular baseline before considering more complex sequence models.

4. **Build an interactive geospatial drought map**  
   Use Folium or Plotly to create a stakeholder-facing output suitable for presentations and portfolios.

### High-value domain extension
5. **Integrate crop calendar data**  
   Weight drought severity by agricultural growing season to make the signal more relevant for food security applications.

### Advanced extension
6. **Add Sentinel-2 for selected agricultural pilot zones**  
   Use higher-resolution imagery once the MODIS-based workflow is stable.

### What is deliberately not prioritized yet
- **LSTM / deep learning** is not the next best step for this notebook.  
  A cleaner real-data baseline with explainable models is more valuable than adding complexity too early.

---

## Portfolio positioning statement

This project can be presented as:

> **An end-to-end geospatial machine learning prototype for drought risk mapping in the MENA region, designed for later integration with real MODIS, rainfall, and crop-calendar data.**

That framing is clear, credible, and strong for employers.

---

**Author:** Ines Gasmi, PhD  
**Project theme:** Climate risk · Drought monitoring · GeoAI foundations · Applied machine learning
